# Exercise 5

Optional study: evolution of consumption.


In [ ]:
import locale
from pathlib import Path

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

try:
    locale.setlocale(locale.LC_ALL, 'fr_FR.utf8')
except locale.Error:
    pass

production_data_path = 'edunao-files/Donnees_France_2024_fr_cet.csv'
capacity_data_path = 'edunao-files/Capacites_France_2024.csv'
temperature_data_path = 'edunao-files/Temperature_Paris_2024_open_meteo.csv'

france_2024_data = pd.read_csv(production_data_path, sep=';', decimal=',', index_col=0)
france_2024_data.index = pd.to_datetime(france_2024_data.index, utc=True).tz_convert('Europe/Paris')
france_2024_data = france_2024_data.apply(pd.to_numeric, errors='coerce')

installed_capacity_data = pd.read_csv(capacity_data_path, sep=';', decimal=',', index_col=0)
installed_capacity_data.index = pd.to_datetime(installed_capacity_data.index, utc=True).tz_convert('Europe/Paris')
installed_capacity_data = installed_capacity_data.apply(pd.to_numeric, errors='coerce')

must_run_columns = ['Wind Onshore', 'Solar', 'Hydro Run-of-river and poundage']
renewable_variable_columns = ['Wind Onshore', 'Solar']
gross_consumption_mw = france_2024_data['Load']
hydro_run_of_river_mw = france_2024_data['Hydro Run-of-river and poundage'].fillna(0)
wind_solar_mw = france_2024_data[renewable_variable_columns].fillna(0).sum(axis=1)
must_run_production_mw = hydro_run_of_river_mw + wind_solar_mw
base_net_consumption_mw = gross_consumption_mw - must_run_production_mw

base_plant_data = pd.DataFrame(
    {
        'Efficiency': [0.35, 0.40, 0.60, 0.35],
        'CO2 intensity (t/MWh_e)': [0.0, 1.0, 0.4, 0.6],
        'Capex (MEUR/MW_e)': [6.0, 1.5, 1.0, 0.7],
        'Fuel cost (EUR/MWh_th)': [5.0, 10.0, 30.0, 50.0],
    },
    index=['Nuclear', 'Coal', 'Gas CCGT', 'Oil OCGT'],
)

annuity_rate = 0.08
base_co2_price_eur_per_t = 80
hours_per_year = len(france_2024_data)

Month = mdates.MonthLocator(bymonthday=1)
MonthFmt = mdates.DateFormatter('%b')


def prepare_plant_data(fuel_costs=None, co2_price_eur_per_t=base_co2_price_eur_per_t):
    plant_data = base_plant_data.copy()
    if fuel_costs is not None:
        for plant, value in fuel_costs.items():
            plant_data.loc[plant, 'Fuel cost (EUR/MWh_th)'] = value

    plant_data['Annual investment cost (EUR/MW/year)'] = (
        plant_data['Capex (MEUR/MW_e)'] * 1_000_000 * annuity_rate
    )
    plant_data['Fuel cost (EUR/MWh_e)'] = plant_data['Fuel cost (EUR/MWh_th)'] / plant_data['Efficiency']
    plant_data['CO2 cost (EUR/MWh_e)'] = plant_data['CO2 intensity (t/MWh_e)'] * co2_price_eur_per_t
    plant_data['Variable cost (EUR/MWh_e)'] = plant_data['Fuel cost (EUR/MWh_e)'] + plant_data['CO2 cost (EUR/MWh_e)']
    return plant_data


def build_screening_segments(plant_data, number_of_hours):
    operating_hours = np.arange(0, number_of_hours + 1)
    screening_costs = pd.DataFrame(index=operating_hours)
    for plant in plant_data.index:
        screening_costs[plant] = (
            plant_data.loc[plant, 'Annual investment cost (EUR/MW/year)']
            + plant_data.loc[plant, 'Variable cost (EUR/MWh_e)'] * operating_hours
        )

    cheapest_plant = screening_costs.idxmin(axis=1)
    segments = []
    segment_start = 0
    current_plant = cheapest_plant.iloc[0]
    for hour, plant in cheapest_plant.iloc[1:].items():
        if plant != current_plant:
            segments.append((segment_start, hour - 1, current_plant))
            segment_start = hour
            current_plant = plant
    segments.append((segment_start, number_of_hours, current_plant))
    return segments, screening_costs


def optimize_fleet(net_consumption_mw, plant_data):
    duration_curve_mw = net_consumption_mw.sort_values(ascending=False).reset_index(drop=True)
    segments, screening_costs = build_screening_segments(plant_data, len(duration_curve_mw))

    def load_at_duration(hours):
        if hours <= 0:
            return float(duration_curve_mw.iloc[0])
        if hours >= len(duration_curve_mw):
            return 0.0
        return float(duration_curve_mw.iloc[int(np.ceil(hours))])

    capacities_mw = pd.Series(0.0, index=plant_data.index)
    for start_hour, end_hour, plant in segments:
        upper_load = load_at_duration(start_hour)
        lower_load = load_at_duration(end_hour + 1)
        capacities_mw.loc[plant] += max(upper_load - lower_load, 0.0)

    return capacities_mw, segments, screening_costs


def dispatch_by_plant(load_mw, capacities_mw, plant_data):
    dispatch = pd.DataFrame(0.0, index=load_mw.index, columns=plant_data.index)
    remaining = np.maximum(load_mw.to_numpy(dtype=float), 0.0)

    for plant in ['Nuclear', 'Coal', 'Gas CCGT', 'Oil OCGT']:
        available = capacities_mw.get(plant, 0.0)
        generation = np.minimum(remaining, available)
        dispatch[plant] = generation
        remaining -= generation

    if np.nanmax(remaining) > 1e-6:
        raise ValueError('The dispatchable fleet is too small for this demand profile.')

    return dispatch


def summarize_fleet(net_consumption_mw, plant_data):
    capacities_mw, segments, _ = optimize_fleet(net_consumption_mw, plant_data)
    dispatch_mw = dispatch_by_plant(net_consumption_mw, capacities_mw, plant_data)
    energy_mwh = dispatch_mw.sum()
    costs_eur = pd.DataFrame(index=plant_data.index)
    costs_eur['Capacity (GW)'] = capacities_mw / 1000
    costs_eur['Energy (TWh)'] = energy_mwh / 1_000_000
    costs_eur['Investment cost (bn EUR/year)'] = (
        capacities_mw * plant_data['Annual investment cost (EUR/MW/year)'] / 1_000_000_000
    )
    costs_eur['Fuel cost (bn EUR/year)'] = energy_mwh * plant_data['Fuel cost (EUR/MWh_e)'] / 1_000_000_000
    costs_eur['CO2 cost (bn EUR/year)'] = energy_mwh * plant_data['CO2 cost (EUR/MWh_e)'] / 1_000_000_000
    costs_eur['Total cost (bn EUR/year)'] = costs_eur[
        ['Investment cost (bn EUR/year)', 'Fuel cost (bn EUR/year)', 'CO2 cost (bn EUR/year)']
    ].sum(axis=1)
    return capacities_mw, dispatch_mw, costs_eur, segments


## 5.1 Renewable production scenarios

Evaluate the effect of increasing onshore wind and photovoltaic production by multiplying their 2024 hourly profiles. Run-of-river hydro is kept unchanged.


In [ ]:
renewable_multipliers = [1, 2, 3, 4]
renewable_scenario_results = []
renewable_duration_curves_gw = {}
base_plant_data = prepare_plant_data()

for multiplier in renewable_multipliers:
    scenario_net_mw = gross_consumption_mw - hydro_run_of_river_mw - multiplier * wind_solar_mw
    curtailed_energy_mwh = np.maximum(-scenario_net_mw, 0).sum()
    residual_energy_mwh = np.maximum(scenario_net_mw, 0).sum()
    capacities_mw, _, costs_eur, _ = summarize_fleet(scenario_net_mw.clip(lower=0), base_plant_data)

    renewable_scenario_results.append(
        {
            'Wind and PV multiplier': multiplier,
            'Residual demand (TWh)': residual_energy_mwh / 1_000_000,
            'Surplus before curtailment (TWh)': curtailed_energy_mwh / 1_000_000,
            'Minimum net demand (GW)': scenario_net_mw.min() / 1000,
            'Peak residual demand (GW)': scenario_net_mw.max() / 1000,
            'Optimal dispatchable capacity (GW)': capacities_mw.sum() / 1000,
            'Total modeled cost (bn EUR/year)': costs_eur['Total cost (bn EUR/year)'].sum(),
        }
    )
    renewable_duration_curves_gw[f'x{multiplier} wind+PV'] = (
        scenario_net_mw.sort_values(ascending=False).reset_index(drop=True) / 1000
    )

renewable_scenario_results = pd.DataFrame(renewable_scenario_results).round(2)
display(renewable_scenario_results)

fig, ax = plt.subplots(figsize=(8, 4))
duration_hours = np.arange(1, hours_per_year + 1)
for label, duration_curve in renewable_duration_curves_gw.items():
    ax.plot(duration_hours, duration_curve, linewidth=.8, label=label)
plt.title('Net Load Duration Curves with Increased Wind and PV')
plt.xlabel('Hours with net load greater than or equal to this value')
plt.ylabel('Net demand (GW)')
ax.set_xlim(1, hours_per_year)
plt.grid(True)
plt.legend()
fig.tight_layout()
fig.savefig('figures/exercise_5-1.pdf')
plt.show()

print('Comment:')
print('- Increasing wind and PV lowers annual residual demand and reduces the middle and low-load parts of the duration curve.')
print('- Peak residual demand falls much less than annual energy demand because wind and solar are not guaranteed during peak hours.')
print('- At high multipliers, negative net demand appears, which means that curtailment, exports, storage, or flexible demand would be required.')


## 5.2 Fuel and carbon price sensitivity

Change fuel and CO2 prices and evaluate the effect on the least-cost dispatchable fleet.


In [ ]:
price_scenarios = {
    'Base': {'co2_price': 80, 'fuel_costs': {}},
    'Low CO2 price': {'co2_price': 20, 'fuel_costs': {}},
    'High CO2 price': {'co2_price': 150, 'fuel_costs': {}},
    'High gas price': {'co2_price': 80, 'fuel_costs': {'Gas CCGT': 80}},
    'High fuel prices': {'co2_price': 100, 'fuel_costs': {'Nuclear': 8, 'Coal': 18, 'Gas CCGT': 70, 'Oil OCGT': 80}},
}

sensitivity_capacities = []
sensitivity_costs = []
interval_tables = {}

for scenario, assumptions in price_scenarios.items():
    plant_data = prepare_plant_data(
        fuel_costs=assumptions['fuel_costs'],
        co2_price_eur_per_t=assumptions['co2_price'],
    )
    capacities_mw, dispatch_mw, costs_eur, segments = summarize_fleet(base_net_consumption_mw.clip(lower=0), plant_data)
    interval_tables[scenario] = pd.DataFrame(segments, columns=['From hour', 'To hour', 'Least-cost plant'])
    for plant, capacity in capacities_mw.items():
        sensitivity_capacities.append({'Scenario': scenario, 'Technology': plant, 'Capacity (GW)': capacity / 1000})
    sensitivity_costs.append(
        {
            'Scenario': scenario,
            'CO2 price (EUR/tCO2)': assumptions['co2_price'],
            'Total capacity (GW)': capacities_mw.sum() / 1000,
            'Total cost (bn EUR/year)': costs_eur['Total cost (bn EUR/year)'].sum(),
            'Average cost (EUR/MWh)': costs_eur['Total cost (bn EUR/year)'].sum() * 1_000_000_000 / base_net_consumption_mw.clip(lower=0).sum(),
        }
    )

sensitivity_capacity_table = pd.DataFrame(sensitivity_capacities).pivot(
    index='Scenario', columns='Technology', values='Capacity (GW)'
).fillna(0).round(2)
sensitivity_cost_table = pd.DataFrame(sensitivity_costs).set_index('Scenario').round(2)

display(sensitivity_capacity_table)
display(sensitivity_cost_table)

fig, ax = plt.subplots(figsize=(9, 4.5))
sensitivity_capacity_table[['Nuclear', 'Coal', 'Gas CCGT', 'Oil OCGT']].plot(kind='bar', stacked=True, ax=ax)
plt.title('Optimal Dispatchable Capacity Under Fuel and Carbon Price Scenarios')
plt.xlabel('Scenario')
plt.ylabel('Capacity (GW)')
plt.xticks(rotation=20, ha='right')
plt.grid(True, axis='y')
plt.legend(title='Technology')
fig.tight_layout()
fig.savefig('figures/exercise_5-2.pdf')
plt.show()

print('Comment:')
print('- Coal enters the optimal fleet only when its fixed and variable costs are low enough relative to gas and nuclear.')
print('- A higher carbon price penalizes coal strongly and favors gas over coal for intermediate operation.')
print('- Higher gas prices increase the cost of the intermediate residual-demand layer and can change the boundary between gas and nuclear.')


## 5.3 Demand curve modification

Modify the consumption curve by adding the electricity demand of a fully electrified light-vehicle fleet. Two charging profiles are compared: flat charging and smart charging in the lowest net-demand hours of each day.


In [ ]:
vehicle_count = 40_000_000
annual_distance_km_per_vehicle = 11_700
vehicle_consumption_kwh_per_100km = 18
annual_ev_energy_mwh = vehicle_count * annual_distance_km_per_vehicle * vehicle_consumption_kwh_per_100km / 100 / 1000

flat_ev_charging_mw = pd.Series(annual_ev_energy_mwh / hours_per_year, index=gross_consumption_mw.index)

# Flexible charging is modeled as daily valley filling. For each day, the EV energy
# is placed so that the daily net-demand profile is as flat as possible.
smart_ev_charging_mw = pd.Series(0.0, index=gross_consumption_mw.index)
daily_groups = list(base_net_consumption_mw.groupby(base_net_consumption_mw.index.date))
daily_ev_energy_mwh = annual_ev_energy_mwh / len(daily_groups)

for _, daily_net in daily_groups:
    values = daily_net.to_numpy(dtype=float)
    sorted_values = np.sort(values)
    target_level = sorted_values[-1]

    for rank in range(1, len(sorted_values) + 1):
        candidate_level = sorted_values[rank] if rank < len(sorted_values) else np.inf
        energy_to_candidate = (candidate_level - sorted_values[:rank]).sum()
        if energy_to_candidate >= daily_ev_energy_mwh:
            target_level = sorted_values[rank - 1] + (
                daily_ev_energy_mwh - (sorted_values[rank - 1] - sorted_values[:rank - 1]).sum()
            ) / rank
            break
    else:
        target_level = sorted_values[-1] + (daily_ev_energy_mwh - (sorted_values[-1] - sorted_values).sum()) / len(sorted_values)

    daily_charge = np.maximum(target_level - values, 0.0)
    smart_ev_charging_mw.loc[daily_net.index] += daily_charge

demand_scenarios = {
    'Base net demand': base_net_consumption_mw,
    'With EV flat charging': base_net_consumption_mw + flat_ev_charging_mw,
    'With EV valley filling': base_net_consumption_mw + smart_ev_charging_mw,
}

demand_modification_results = []
for scenario, demand_mw in demand_scenarios.items():
    capacities_mw, _, costs_eur, _ = summarize_fleet(demand_mw.clip(lower=0), base_plant_data)
    demand_modification_results.append(
        {
            'Scenario': scenario,
            'Annual net energy (TWh)': demand_mw.clip(lower=0).sum() / 1_000_000,
            'Average net demand (GW)': demand_mw.mean() / 1000,
            'Peak net demand (GW)': demand_mw.max() / 1000,
            'Minimum net demand (GW)': demand_mw.min() / 1000,
            'Optimal dispatchable capacity (GW)': capacities_mw.sum() / 1000,
            'Total modeled cost (bn EUR/year)': costs_eur['Total cost (bn EUR/year)'].sum(),
        }
    )

demand_modification_results = pd.DataFrame(demand_modification_results).round(2)
display(demand_modification_results)

fig, ax = plt.subplots(figsize=(8, 4))
duration_hours = np.arange(1, hours_per_year + 1)
for scenario, demand_mw in demand_scenarios.items():
    duration_curve_gw = demand_mw.sort_values(ascending=False).reset_index(drop=True) / 1000
    ax.plot(duration_hours, duration_curve_gw, linewidth=.8, label=scenario)
plt.title('Effect of Electric Vehicle Charging on Net Load Duration Curves')
plt.xlabel('Hours with net load greater than or equal to this value')
plt.ylabel('Net demand (GW)')
ax.set_xlim(1, hours_per_year)
plt.grid(True)
plt.legend()
fig.tight_layout()
fig.savefig('figures/exercise_5-3.pdf')
plt.show()

print('Comment:')
print(f'- Electrifying the full light-vehicle fleet adds {annual_ev_energy_mwh / 1_000_000:.1f} TWh/year of electricity demand.')
print('- Flat charging raises the whole duration curve by a nearly constant amount.')
print('- Valley-filling charging places daily EV energy in low-demand hours and therefore limits the increase in peak net demand.')


## 5.4 Temperature-consumption correlation

Estimate the correlation between consumption and temperature using hourly Paris 2 m temperature from the Open-Meteo historical archive. Paris temperature is used as a simple weather proxy for France, so the result is only an order-of-magnitude indicator.


In [ ]:
temperature_raw = pd.read_csv(temperature_data_path, skiprows=3)
temperature_c = pd.Series(
    temperature_raw['temperature_2m (°C)'].to_numpy(dtype=float),
    index=france_2024_data.index,
    name='Paris temperature (°C)',
)

weather_load = pd.DataFrame(
    {
        'Load (GW)': gross_consumption_mw / 1000,
        'Temperature (°C)': temperature_c,
    }
)
daily_weather_load = weather_load.resample('D').mean()

pearson_hourly = weather_load['Load (GW)'].corr(weather_load['Temperature (°C)'])
pearson_daily = daily_weather_load['Load (GW)'].corr(daily_weather_load['Temperature (°C)'])

cold_days = daily_weather_load[daily_weather_load['Temperature (°C)'] <= 15]
heating_slope_gw_per_celsius, heating_intercept = np.polyfit(
    cold_days['Temperature (°C)'],
    cold_days['Load (GW)'],
    1,
)

correlation_results = pd.DataFrame(
    {
        'Value': [pearson_hourly, pearson_daily, heating_slope_gw_per_celsius, -heating_slope_gw_per_celsius],
        'Unit': ['-', '-', 'GW/°C', 'GW/°C below 15 °C'],
    },
    index=[
        'Hourly Pearson correlation',
        'Daily Pearson correlation',
        'Cold-weather regression slope',
        'Heating sensitivity magnitude',
    ],
)

display(correlation_results.round(3))

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.scatter(
    daily_weather_load['Temperature (°C)'],
    daily_weather_load['Load (GW)'],
    s=12,
    alpha=.6,
    label='Daily averages',
)
fit_temperatures = np.linspace(cold_days['Temperature (°C)'].min(), 15, 50)
ax.plot(
    fit_temperatures,
    heating_slope_gw_per_celsius * fit_temperatures + heating_intercept,
    color='tab:red',
    linewidth=1.5,
    label='Linear fit for T <= 15 °C',
)
plt.title('Correlation Between Electricity Consumption and Temperature')
plt.xlabel('Daily mean Paris temperature (°C)')
plt.ylabel('Daily mean French load (GW)')
plt.grid(True)
plt.legend()
fig.tight_layout()
fig.savefig('figures/exercise_5-4.pdf')
plt.show()

print('Comment:')
print(f'- The daily Pearson correlation is {pearson_daily:.2f}, confirming higher electricity demand during colder periods.')
print(f'- For days colder than 15 °C, the fitted slope is {heating_slope_gw_per_celsius:.2f} GW/°C.')
print(f'- This corresponds to an approximate heating sensitivity of {-heating_slope_gw_per_celsius:.2f} GW for each 1 °C temperature decrease.')
